# 1940 - 2014 US UAP sightings

## EDA and clean data
- dates 1940 - 2014 
- US sightings only 
- data cleaning 
- organize columns to match ERD
- save csv  

## calculate geohash_5, 6, 7 based on latitude and longitude

## add weather data in batched
- weather_api.py 
    - calls to openmeteo (batches)
    - 8,000 calls / day limit
    - save each batch to csv until all rows have weather data

## perform additional analysis on csv with weather data 
- columns that need weather data ( example conditions group )
- save final csv
- * create realtional database with SQLite 


In [188]:
import pandas as pd
import datetime
import pygeohash as pgh

from python.geo_location import create_geohashes 

In [59]:
uap_df = pd.read_csv("data/complete.csv", usecols=range(0, 11), low_memory=False)


In [60]:
uap_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88875 entries, 0 to 88874
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime              88875 non-null  object 
 1   city                  88679 non-null  object 
 2   state                 81356 non-null  object 
 3   country               76314 non-null  object 
 4   shape                 85757 non-null  object 
 5   duration (seconds)    88873 non-null  object 
 6   duration (hours/min)  85772 non-null  object 
 7   comments              88749 non-null  object 
 8   date posted           88875 non-null  object 
 9   latitude              88875 non-null  object 
 10  longitude             88875 non-null  float64
dtypes: float64(1), object(10)
memory usage: 7.5+ MB


In [61]:
uap_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611


In [62]:
# At least one row in the 'datetime' col contains an invalid 24:00 for the time
# AI Use: Grok assistance in fixing any entries with 24:00 in formating datetime

# Make a copy of the original column
col = uap_df['datetime'].astype(str)

# Handle 24:00 cases
mask_24 = col.str.contains('24:00', na=False)
col = col.str.replace('24:00', '00:00', regex=False)

# Convert to real datetime
uap_df['datetime'] = pd.to_datetime(col, format='%m/%d/%Y %H:%M', errors='coerce')

# Fix the date rollover for 24:00
uap_df.loc[mask_24, 'datetime'] = uap_df.loc[mask_24, 'datetime'] + pd.Timedelta(days=1)

# === Create the columns you want ===
uap_df['datetime_formatted'] = uap_df['datetime'].dt.strftime('%Y-%m-%d %H:%M')   # yyyy-mm-dd hh:mm
uap_df['full_date']           = uap_df['datetime'].dt.strftime('%Y-%m-%d')         # yyyy-mm-dd only


In [63]:
uap_df['datetime'].info

<bound method Series.info of 0       1949-10-10 20:30:00
1       1949-10-10 21:00:00
2       1955-10-10 17:00:00
3       1956-10-10 21:00:00
4       1960-10-10 20:00:00
                ...        
88870   2013-09-09 22:00:00
88871   2013-09-09 22:20:00
88872   2013-09-09 23:00:00
88873   2013-09-09 23:00:00
88874   2013-09-09 23:30:00
Name: datetime, Length: 88875, dtype: datetime64[ns]>

In [ ]:
us_uap_df = uap_df[uap_df["country"].str.lower() == "us"].copy()
us_uap_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,datetime_formatted,full_date
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111,1949-10-10 20:30,1949-10-10
3,1956-10-10 21:00:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833,1956-10-10 21:00,1956-10-10
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611,1960-10-10 20:00,1960-10-10
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,5 minutes,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.5950000,-82.188889,1961-10-10 19:00,1961-10-10
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,20 minutes,A bright orange color changing to reddish colo...,10/2/1999,41.1175000,-73.408333,1965-10-10 23:45,1965-10-10


In [86]:
us_uap_after_1940_df = us_uap_df[
    us_uap_df["datetime"] >= '1940-01-01'
    
].copy()

us_uap_after_1940_df.info()



<class 'pandas.core.frame.DataFrame'>
Index: 70276 entries, 0 to 88874
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              70276 non-null  datetime64[ns]
 1   city                  70276 non-null  object        
 2   state                 70276 non-null  object        
 3   country               70276 non-null  object        
 4   shape                 68047 non-null  object        
 5   duration (seconds)    70275 non-null  object        
 6   duration (hours/min)  68054 non-null  object        
 7   comments              70248 non-null  object        
 8   date posted           70276 non-null  object        
 9   latitude              70276 non-null  object        
 10  longitude             70276 non-null  float64       
 11  datetime_formatted    70276 non-null  object        
 12  full_date             70276 non-null  object        
dtypes: datetime64[ns](1),

In [ ]:
# fix null or na values

us_uap_after_1940_df.isnull().sum()





datetime                   0
city                       0
state                      0
country                    0
shape                   2229
duration (seconds)         1
duration (hours/min)    2222
comments                  28
date posted                0
latitude                   0
longitude                  0
datetime_formatted         0
full_date                  0
dtype: int64

In [ ]:
# fix shape null values
# unique values of shape

us_uap_after_1940_df['shape'].unique()




array(['cylinder', 'circle', 'light', 'sphere', 'disk', 'fireball',
       'unknown', 'oval', 'other', 'rectangle', 'chevron', 'formation',
       'triangle', 'cigar', nan, 'delta', 'changing', 'diamond', 'flash',
       'egg', 'teardrop', 'cone', 'cross', 'pyramid', 'round', 'flare',
       'hexagon', 'crescent', 'changed'], dtype=object)

In [ ]:
# shape null and nan values as unknown 

us_uap_after_1940_df['shape'] = us_uap_after_1940_df['shape'].fillna('unknown')
us_uap_after_1940_df.isnull().sum()

datetime                   0
city                       0
state                      0
country                    0
shape                      0
duration (seconds)         1
duration (hours/min)    2222
comments                  28
date posted                0
latitude                   0
longitude                  0
datetime_formatted         0
full_date                  0
dtype: int64

In [ ]:
us_uap_after_1940_df['shape'].unique()

array(['cylinder', 'circle', 'light', 'sphere', 'disk', 'fireball',
       'unknown', 'oval', 'other', 'rectangle', 'chevron', 'formation',
       'triangle', 'cigar', 'delta', 'changing', 'diamond', 'flash',
       'egg', 'teardrop', 'cone', 'cross', 'pyramid', 'round', 'flare',
       'hexagon', 'crescent', 'changed'], dtype=object)

In [ ]:
# fix durration (sec) null value - only one
us_uap_after_1940_df[us_uap_after_1940_df['duration (seconds)'].isnull()]

# airport sighting, unclear durration based on other values - drop row in place 
us_uap_after_1940_df.dropna(subset=['duration (seconds)'], inplace=True)

# check null value totals 
us_uap_after_1940_df.isnull().sum()

datetime                   0
city                       0
state                      0
country                    0
shape                      0
duration (seconds)         0
duration (hours/min)    2222
comments                  28
date posted                0
latitude                   0
longitude                  0
datetime_formatted         0
full_date                  0
dtype: int64

In [ ]:
# remove duration hours / mins (null and wierd values + seconds column gives diration and is more nomalized)
us_uap_after_1940_df.drop(columns = ['duration (hours/min)'], inplace=True)

# check null value totals 
us_uap_after_1940_df.isnull().sum()

datetime               0
city                   0
state                  0
country                0
shape                  0
duration (seconds)     0
comments              28
date posted            0
latitude               0
longitude              0
datetime_formatted     0
full_date              0
dtype: int64

In [ ]:
# fill null values in 'comments'

# what do the rows look like? 
us_uap_after_1940_df[us_uap_after_1940_df['comments'].isnull()]

# fill with null comments with 'no comment' 
us_uap_after_1940_df['comments'] = us_uap_after_1940_df['comments'].fillna('no comments')

us_uap_after_1940_df.isnull().sum()


datetime              0
city                  0
state                 0
country               0
shape                 0
duration (seconds)    0
comments              0
date posted           0
latitude              0
longitude             0
datetime_formatted    0
full_date             0
dtype: int64

In [130]:
# convert durration to int
# issues: some entries have non intergers in them
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(float)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].round(0)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(int)
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111,1949-10-10 20:30,1949-10-10
3,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833,1956-10-10 21:00,1956-10-10
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611,1960-10-10 20:00,1960-10-10
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.5950000,-82.188889,1961-10-10 19:00,1961-10-10
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.1175000,-73.408333,1965-10-10 23:45,1965-10-10


In [ ]:
# convert latitude to float
us_uap_after_1940_df['latitude'] = us_uap_after_1940_df['latitude'].astype(float)

us_uap_after_1940_df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration (seconds)             int64
comments                      object
date posted                   object
latitude                     float64
longitude                    float64
datetime_formatted            object
full_date                     object
dtype: object

In [ ]:
# full_date as datetime

us_uap_after_1940_df['full_date'] = pd.to_datetime(us_uap_after_1940_df['full_date'])
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949-10-10 20:30,1949-10-10
3,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956-10-10 21:00,1956-10-10
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960-10-10 20:00,1960-10-10
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961-10-10 19:00,1961-10-10
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965-10-10 23:45,1965-10-10


In [ ]:
# year (int), month(int) from full_date

us_uap_after_1940_df['year'] = us_uap_after_1940_df['full_date'].astype(str).str[:4]
us_uap_after_1940_df['year'] = us_uap_after_1940_df['year'].astype(int)

us_uap_after_1940_df['month'] = us_uap_after_1940_df['full_date'].astype(str).str[5:7]
us_uap_after_1940_df['month'] = us_uap_after_1940_df['month'].astype(int)

us_uap_after_1940_df.head()
us_uap_after_1940_df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration (seconds)             int64
comments                      object
date posted                   object
latitude                     float64
longitude                    float64
datetime_formatted            object
full_date             datetime64[ns]
year                           int64
month                          int64
dtype: object

In [145]:
# season (str) from 'month'

season_groups = {
    '01': 'winter', 
    '02': 'winter', 
    '03': 'spring', 
    '04': 'spring', 
    '05': 'spring',
    '06': 'summer',
    '07': 'summer', 
    '08': 'summer',
    '09': 'fall', 
    '10': 'fall',
    '11': 'fall',
    '12': 'winter',
}

us_uap_after_1940_df['season'] = us_uap_after_1940_df['month'].astype(str).str.zfill(2).map(season_groups)

In [146]:
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date,year,month,season
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949-10-10 20:30,1949-10-10,1949,10,fall
3,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956-10-10 21:00,1956-10-10,1956,10,fall
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960-10-10 20:00,1960-10-10,1960,10,fall
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961-10-10 19:00,1961-10-10,1961,10,fall
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965-10-10 23:45,1965-10-10,1965,10,fall


In [152]:
# add in kp_index
# dataframe from kp_index.csv

kp_df = pd.read_csv("data/kp_index.csv", usecols=('datetime', 'kp', 'ap'))
kp_df.head(20)

,kp,ap,datetime
0,3.333,18,1932-01-01 00:00:00
1,2.667,12,1932-01-01 03:00:00
2,2.333,9,1932-01-01 06:00:00
3,2.667,12,1932-01-01 09:00:00
4,3.333,18,1932-01-01 12:00:00
5,2.667,12,1932-01-01 15:00:00
6,3.333,18,1932-01-01 18:00:00
7,3.333,18,1932-01-01 21:00:00
8,3.667,22,1932-01-02 00:00:00
9,3.667,22,1932-01-02 03:00:00


In [ ]:
# ensure datetime column is a datetime type
kp_df['datetime'] = pd.to_datetime(kp_df['datetime'])
kp_df.dtypes

kp                 float64
ap                   int64
datetime    datetime64[ns]
dtype: object

In [ ]:
# kp table includes values in utz (5-6 hours ahead of most US timezones)
# most uap sightings in the US happen late in the evening / close 9:00pm or 21:00 
# 03:00:00 or 3am UTZ would be the best match for most sightings .. which is 9pm central standard time 

# only 03:00 utz (it will have the next day's date, so we would need to convert to US time zome)
target_time = datetime.time(3, 0, 0)

kp_3am_utz_df = kp_df[kp_df['datetime'].dt.time == target_time].copy()

# subtract 6 hours to convert UTZ to US Central Standard Time

kp_9pm_us_cst_df = kp_3am_utz_df.copy()
kp_9pm_us_cst_df['datetime'] = kp_9pm_us_cst_df['datetime'] - pd.Timedelta(hours=6)
kp_9pm_us_cst_df.head()



,kp,ap,datetime
1,2.667,12,1931-12-31 21:00:00
9,3.667,22,1932-01-01 21:00:00
17,3.333,18,1932-01-02 21:00:00
25,0.333,2,1932-01-03 21:00:00
33,0.000,0,1932-01-04 21:00:00


In [171]:
kp_9pm_us_cst_df['full_date'] = kp_9pm_us_cst_df['datetime'].dt.strftime('%Y-%m-%d')
kp_9pm_us_cst_df['full_date'] = pd.to_datetime(kp_9pm_us_cst_df['full_date'])
kp_9pm_us_cst_df.head()

,kp,ap,datetime,full_date
1,2.667,12,1931-12-31 21:00:00,1931-12-31
9,3.667,22,1932-01-01 21:00:00,1932-01-01
17,3.333,18,1932-01-02 21:00:00,1932-01-02
25,0.333,2,1932-01-03 21:00:00,1932-01-03
33,0.000,0,1932-01-04 21:00:00,1932-01-04


In [172]:
# join to full_date to us uap sightings

us_uap_after_1940_df = pd.merge(us_uap_after_1940_df, kp_9pm_us_cst_df[['full_date','kp', 'ap']], on='full_date', how='left')


In [173]:
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date,year,month,season,kp,ap
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949-10-10 20:30,1949-10-10,1949,10,fall,2.667,12
1,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956-10-10 21:00,1956-10-10,1956,10,fall,2.667,12
2,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960-10-10 20:00,1960-10-10,1960,10,fall,3.667,22
3,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961-10-10 19:00,1961-10-10,1961,10,fall,1.667,6
4,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965-10-10 23:45,1965-10-10,1965,10,fall,0.333,2


In [177]:
# create state_code column in uppercase to match bigfoot data and ERD

us_uap_after_1940_df['state_code'] = us_uap_after_1940_df['state'].str.upper()

# rename to match ERD

us_uap_after_1940_df = us_uap_after_1940_df.rename(
    columns={
        'duration (seconds)': 'duration_secs',
        'kp': 'solar_kp_index', 
        'ap': 'solar_ap_index'
    }
)

In [178]:
us_uap_after_1940_df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration_secs                  int64
comments                      object
date posted                   object
latitude                     float64
longitude                    float64
datetime_formatted            object
full_date             datetime64[ns]
year                           int64
month                          int64
season                        object
solar_kp_index               float64
solar_ap_index                 int64
state_code                    object
dtype: object

In [192]:
# calculate shape_group



# group shapes by types 

shape_groups = {
    'circle': 'round', 'sphere': 'round', 'disk': 'round', 
    'oval': 'round', 'round': 'round', 'crescent': 'round', 'dome': 'round',
    'egg': 'round', 'teardrop': 'round',
    
    'triangle': 'triangle', 'delta': 'triangle', 'chevron': 'triangle', 
    'pyramid': 'triangle', 'diamond': 'triangle',
    
    'cylinder': 'cigar', 'cigar': 'cigar',
    
    'light': 'light', 'fireball': 'light', 'flash': 'light', 'flare': 'light',
    
    'changing': 'changing', 'changed': 'changing', 'formation': 'changing', 'cone': 'changing',
    'cross': 'changing', 'hexagon': 'changing', 'rectangle': 'changing',
    
    'other': 'other', 
    'nan': 'other',
    'unknown': 'other',
    '': 'other'
}

us_uap_after_1940_df['shape_group'] = us_uap_after_1940_df['shape'].map(shape_groups)

us_uap_after_1940_df['shape_group'].value_counts()

shape_group
light       20796
round       20253
other       12139
triangle     8809
changing     5442
cigar        2836
Name: count, dtype: int64

In [193]:
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration_secs,comments,date posted,latitude,longitude,...,year,month,season,solar_kp_index,solar_ap_index,state_code,shape_group,geohash_7,geohash_6,geohash_5
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,...,1949,10,fall,2.667,12,TX,cigar,9v66521,9v6652,9v665
1,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,...,1956,10,fall,2.667,12,TX,round,9v5kbg2,9v5kbg,9v5kb
2,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,...,1960,10,fall,3.667,22,HI,light,87zcc54,87zcc5,87zcc
3,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,...,1961,10,fall,1.667,6,TN,round,dnt81tz,dnt81t,dnt81
4,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,...,1965,10,fall,0.333,2,CT,round,dr7cct1,dr7cct,dr7cc


In [194]:
# save to a csv copy before adding geohash and weather data

us_uap_after_1940_df.to_csv("data/us_uap_1940_v1.csv", index=False)

In [195]:
# create_geohashes function imported from python.geo_location
# create columns for geohash_7, 6, 5 ( use later for plotting, maps, and proximity )

us_uap_after_1940_df[['geohash_7', 'geohash_6', 'geohash_5']] = us_uap_after_1940_df.apply(
    lambda row: create_geohashes(row['latitude'], row['longitude']),
    axis=1,
    result_type='expand'
)

us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration_secs,comments,date posted,latitude,longitude,...,year,month,season,solar_kp_index,solar_ap_index,state_code,shape_group,geohash_7,geohash_6,geohash_5
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,...,1949,10,fall,2.667,12,TX,cigar,9v66521,9v6652,9v665
1,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,...,1956,10,fall,2.667,12,TX,round,9v5kbg2,9v5kbg,9v5kb
2,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,...,1960,10,fall,3.667,22,HI,light,87zcc54,87zcc5,87zcc
3,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,...,1961,10,fall,1.667,6,TN,round,dnt81tz,dnt81t,dnt81
4,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,...,1965,10,fall,0.333,2,CT,round,dr7cct1,dr7cct,dr7cc


In [196]:
# write over US UAP CSV file 
us_uap_after_1940_df.to_csv("data/us_uap_1940_v1.csv", index=False)

Prep for weather api call 

In [204]:
# Add the new columns first (with None values) as filler before API calls 

# use hourly with uap sightings 

# hourly
us_uap_after_1940_df['temperature_f'] = None # previously temperature mid
us_uap_after_1940_df['cloud_cover'] = None
us_uap_after_1940_df['precip_total'] = None
us_uap_after_1940_df['dew_point'] = None 

# calculated (after api call)
us_uap_after_1940_df['precip_type'] = None # calculated from precip_total and temp 

# no lunar data - excetp on proximity matches 
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration_secs,comments,date posted,latitude,longitude,...,state_code,shape_group,geohash_7,geohash_6,geohash_5,temperature_f,cloud_cover,precip_total,dew_point,precip_type
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,...,TX,cigar,9v66521,9v6652,9v665,None,None,None,None,None
1,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,...,TX,round,9v5kbg2,9v5kbg,9v5kb,None,None,None,None,None
2,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,...,HI,light,87zcc54,87zcc5,87zcc,None,None,None,None,None
3,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,...,TN,round,dnt81tz,dnt81t,dnt81,None,None,None,None,None
4,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,...,CT,round,dr7cct1,dr7cct,dr7cc,None,None,None,None,None


In [205]:
us_uap_after_1940_df.to_csv("data/us_uap_1940_v1.csv", index=False)